# Part 10 - Pause, Approve, Resume, Steer: human-in-the-loop on a durable agent

*Agents from First Principles, Part 10.*

Part 9 made the agent durable: it can crash and resume without losing state or double-charging. But it still runs start to finish on its own. Two things a real deployment needs are missing. Some actions must WAIT for a human, a refund over a threshold should not fire until someone approves it, and a person watching the run may want to CORRECT it mid-flight rather than only approve or deny. The durable agent has no way to stop, hand control to a human, and come back.

This part adds that, on top of Part 9's journal. Four moves:

1. **interrupt().** When the agent reaches a gated action (a refund over the approval threshold), it does not call the tool. It journals a `pending_approval` event with a token and RAISES a serializable `PendingApproval` that is caught at the top level. Crucially this is NOT `sys.exit`: the exception carries the token and the pending action, so a notebook (or a server) catches it, persists the journal, and returns the token to the caller. The run is paused, not dead.

2. **resume(run_id, decision).** Later, a human decides. `resume` rehydrates the run by replaying the journal (Part 9), records the decision, and continues. Because the gated refund still carries its idempotency key, an approved resume executes it EXACTLY ONCE even if `resume` is retried.

3. **STEER, not just approve/deny.** The human decision is a first-class action, not a binary gate. An approver can APPROVE, DENY, or STEER: inject a correction (here, lower the refund to an amount under the threshold) that changes what the agent does next. Answering a clarifying question back to the user is the same mechanism.

4. **Streaming progress.** Because every step is a journal event, a live progress feed is just a read over the same log. The pause, the decision, and the resume all show up in the stream.

**Cross-process, honestly.** A real deployment pauses in one process (the request that hit the gate returns the token) and resumes in another (a later request, or a CLI re-invocation, calls `resume` on the same journal). We model both phases in one file by catching `PendingApproval` and then calling `resume` on the persisted world; the CLI two-process version is the same journal read by a second script invocation.

This REUSES Part 9's journal and idempotency key (referenced, not rebuilt) and does not re-teach the reason/act/observe loop. The new thing is the human-in-the-loop checkpoint and the first-class decision.

> **Runs with no network, no API key, and no third-party dependency.** The deterministic plan is the offline default; `generate()` is the real-LLM path one flag away. Set `OPENAI_API_KEY` to see the real banner; the code always falls through to the deterministic plan so output stays reproducible.

Companion script: `pause_approve_resume.py`

## Setup, and the shape of a pause

Two standard-library imports do the work: `os` lets us check for an API key without ever requiring one, and `copy` lets us deep-copy the paused world so the approve, deny, and steer branches each start from the SAME surviving pause. Everything else is plain Python, so every cell runs fully offline, exactly as in Parts 1-9.

Be precise about what "pause" means here, because the honesty is the lesson:

- A pause is NOT a process exit. `interrupt()` raises a normal Python exception (`PendingApproval`) that the top level catches. A notebook keeps its kernel; a server keeps serving. The run is suspended, with its token and pending action carried in the exception.
- "Cross-process resume" means re-entering `resume()` on the same surviving journal. On disk that journal is the artifact a second request (or a CLI re-run) would load.
- The decision a human makes is itself journaled as a first-class event, so APPROVE, DENY, and STEER are all just data in the log, replayed like any other step.

The next cell also fixes the reproducibility knobs, exactly as in Part 9. `RUN_ID` ties every event to the same run across a pause and a resume, and `TS_BASE` plus the sequence number gives each event a frozen timestamp. `THRESHOLD` is the new policy knob: a refund AT OR BELOW it runs unattended, a refund ABOVE it is gated and must wait for a human. The task is a $180 refund, over the $100 threshold, so it will pause.

In [ ]:
import copy
import os

RUN_ID = "run-aa10"                       # a fixed run id so the journal is reproducible
TS_BASE = "2026-07-06T09:00:"             # frozen timestamps: TS_BASE + seq
THRESHOLD = 100.0                         # refunds over this need human approval

print("ready -- no network, no API key, no third-party dependency required")
print(f"run_id {RUN_ID!r}; threshold ${THRESHOLD:.0f}; timestamps frozen as {TS_BASE}NNZ")

## Step 1 - The journal (carried from Part 9), now folding in human decisions

This is Part 9's journal, REUSED, not rebuilt: an append-only event log where state is the FOLD over the log. The `world` still bundles the three durable artifacts that survive a pause: the `journal` (the event log), the `keystore` (idempotency keys the provider has honored), and the `ledger` (money that actually moved). `append` writes one event with the next sequence number, the fixed run id, and a frozen timestamp.

`replay` is the one piece that grows for this part. As in Part 9 it folds the log to learn which steps completed (with their results) and whether the run finished. The new line is `approval_decision`: a recorded human decision is now part of the folded state, keyed by the step index, carrying both the decision (`approve` / `deny` / `steer`) and any correction. On resume the loop reads this fold to know a human has spoken, exactly as it reads `tool_result` to know a step already ran.

In [ ]:
def new_world():
    return {"journal": [], "keystore": {}, "ledger": {}}


def append(world, etype, data):
    seq = len(world["journal"])
    world["journal"].append({"seq": seq, "run_id": RUN_ID, "ts": f"{TS_BASE}{seq:02d}Z",
                             "type": etype, "data": data})


def replay(journal):
    """Fold the log: completed steps + results, the recorded human decisions, and
    whether the run finished. The approval_decision fold is the new piece; the rest
    is Part 9's replay."""
    completed, results, decisions, finished = set(), {}, {}, False
    for e in journal:
        d, t = e["data"], e["type"]
        if t == "tool_result":
            completed.add(d["idx"]); results[d["idx"]] = d["result"]
        elif t == "approval_decision":
            decisions[d["idx"]] = {"decision": d["decision"], "correction": d.get("correction")}
        elif t == "finished":
            finished = True
    return completed, results, decisions, finished


print("new_world, append, replay ready (Part 9's journal, with a decision fold).")

## Step 2 - The interrupt: a serializable exception caught at the top level, NOT `sys.exit`

This is the heart of the part, so state it plainly. When the agent hits the gate it does not kill the process. It raises `PendingApproval`, a normal Python exception that carries three things: a `token` (the handle a human will quote to resume), the pending `action` (the tool and args that are waiting), and a `reason` (why it paused). The top level catches it, persists the journal, and returns the token to the caller.

**Why not `sys.exit`?** Because `sys.exit` would tear down the kernel: a notebook would stop, a web request handler would crash the worker. An exception caught at the top level leaves the host process alive and runnable, which is exactly why the demo below stays runnable cell after cell. The run is suspended, not dead, and because the token and action are plain data, the whole pause is serializable: it can cross a process boundary on disk and be resumed by a second invocation.

**And the decision will be a first-class action, beyond approve/deny.** A binary gate can only say yes or no, but a watching human often knows the RIGHT thing to do, not just whether the wrong thing should fire. By modeling the decision as data (`approve` / `deny` / `steer` plus an optional correction), the human can STEER: inject a correction (here, lower the refund under the threshold) that changes what the agent does next. The same mechanism answers a clarifying question the agent asked back to the user.

In [ ]:
class PendingApproval(Exception):
    """Raised at a gated action. Caught at the TOP LEVEL (never sys.exit), so the
    host process stays alive. Carries the token + pending action so the caller can
    persist and hand off to a human."""

    def __init__(self, token, action, reason):
        super().__init__(f"pending approval: {reason}")
        self.token = token
        self.action = action
        self.reason = reason


print("PendingApproval ready (a serializable interrupt, not a process exit).")

## Step 3 - The tools (from Part 9): a read-only search and an idempotent-by-key refund

These are Part 9's tools, unchanged in spirit. `exec_search` is read-only: it returns the policy clause. `exec_refund` is the side-effecting tool and it is IDEMPOTENT BY KEY: the provider (the keystore) remembers every key it has honored, so the same key never charges twice. That key is exactly what makes an APPROVED resume effectively-once: if `resume` is retried after the refund already posted, the key returns the original result instead of charging again. The gate is new; the effectively-once guarantee underneath it is inherited.

`STEPS` is the same fixed two-step plan as `(tool, args, idempotency_key)` tuples: a read-only `search_policy`, then `process_refund` for `ORD-3300` at `$180.00`. The refund's stable key, `ORD-3300:refund`, is what the provider recognizes on a retried resume; the `$180.00` amount is above the `$100` threshold ON PURPOSE, so it trips the gate and forces the pause.

In [ ]:
def exec_search(args):
    return "Refunds after the window are refundable minus a 10% restocking fee."


def exec_refund(world, order_id, amount, idem_key):
    ks = world["keystore"]
    if idem_key in ks:                                # the provider has seen this key
        return ks[idem_key], True                     # return the original; do NOT re-charge
    world["ledger"][order_id] = world["ledger"].get(order_id, 0.0) + amount
    result = f"refunded ${amount:.2f} to {order_id}"
    ks[idem_key] = result
    return result, False


STEPS = [
    ("search_policy", {"query": "refund policy window"}, None),
    ("process_refund", {"order_id": "ORD-3300", "amount": 180.0}, "ORD-3300:refund"),
]

print(f"{len(STEPS)} steps; refund ${STEPS[1][1]['amount']:.0f} to {STEPS[1][1]['order_id']} "
      f"(over the ${THRESHOLD:.0f} threshold -> will pause)")

## Step 4 - The run loop: the gate, and acting on a recorded decision

`run` is Part 9's replay-driven loop with one new branch at the gated step. It journals `run_started` on a fresh world, folds the journal, and short-circuits if the run already finished. For each step, a completed step is MEMOIZED (its recorded result is printed, not re-run), exactly as in Part 9.

The new logic is the gate. A `process_refund` whose amount exceeds `THRESHOLD` is `gated`. At a gated step the loop checks the folded `decisions`:

- **No decision yet** -> journal `pending_approval` with a token and RAISE `PendingApproval`. This is the interrupt: the run pauses here.
- **deny** -> journal a refusal result and `finished`; no money moves.
- **steer** -> apply the correction (lower the amount), print that it was steered, then fall through and execute the now-under-threshold refund.
- **approve** -> fall through and execute the refund as planned.

Everything that executes (not gated, or approved, or steered) runs the tool and journals a `tool_result`, just like Part 9. The decision is data the loop reads on replay; the loop never re-prompts a human, it acts on what the journal already records.

In [ ]:
def run(world):
    if not world["journal"]:
        append(world, "run_started", {"run_id": RUN_ID})
    completed, results, decisions, finished = replay(world["journal"])
    if finished:
        return

    for idx, (tool, args, idem) in enumerate(STEPS):
        if idx in completed:                          # MEMOIZED: do not re-run (Part 9)
            print(f"    step {idx} {tool}: memoized -> {results[idx]!r}")
            continue

        amount = args.get("amount")
        gated = (tool == "process_refund" and amount > THRESHOLD)
        if gated:
            decision = decisions.get(idx)
            if decision is None:                       # nobody has decided yet: PAUSE
                token = f"appr-{idx}"
                action = {"tool": tool, "args": args}
                append(world, "pending_approval",
                       {"idx": idx, "token": token, "action": action,
                        "reason": f"refund ${amount:.2f} exceeds the ${THRESHOLD:.0f} threshold"})
                raise PendingApproval(token, action,
                                      f"refund ${amount:.2f} exceeds the ${THRESHOLD:.0f} threshold")
            if decision["decision"] == "deny":
                append(world, "tool_result", {"idx": idx, "tool": tool,
                                              "result": "DENIED by approver (no money moved)"})
                append(world, "finished", {"answer": "Refund denied by approver; no money moved."})
                print(f"    step {idx} {tool}: DENIED by approver -> no money moved")
                return
            if decision["decision"] == "steer":        # a correction, not a yes/no
                amount = decision["correction"]["amount"]
                args = {**args, "amount": amount}
                print(f"    step {idx} {tool}: STEERED by approver -> amount lowered to ${amount:.2f} "
                      f"(now under the ${THRESHOLD:.0f} threshold)")

        # execute (not gated, or approved, or steered under threshold)
        if tool == "process_refund":
            result, was_idem = exec_refund(world, args["order_id"], args["amount"], idem)
            tag = " (idempotent)" if was_idem else ""
        else:
            result, was_idem, tag = exec_search(args), False, ""
        append(world, "tool_result", {"idx": idx, "tool": tool, "result": result})
        print(f"    step {idx} {tool}: {result}{tag}")

    final = f"Refund of ${world['ledger'].get('ORD-3300', 0.0):.2f} for ORD-3300 is complete."
    append(world, "finished", {"answer": final})
    print(f"    finished -> {final}")


print("run ready (Part 9's loop + the approval gate).")

## Step 5 - resume(): a human decides, the journal records it, the run continues

`resume` is what a second request (or a CLI re-invocation) calls on the paused world. It does one thing of its own: journal the human's `approval_decision` (the decision plus any correction). Then it re-enters `run()`, which replays the journal, now SEES the decision in the fold, and acts on it. Because the decision is a first-class journal event, APPROVE, DENY, and STEER all flow through the same path; STEER is not a special case bolted on, it is just a decision that carries a correction.

The effectively-once guarantee from Part 9 still holds here: if `resume` were retried after an approved refund already posted, the stable idempotency key would return the original result rather than charging twice.

In [ ]:
def resume(world, decision, correction=None, idx=1):
    """A human decides on a paused run. Record the decision as a journal event, then
    re-enter run(), which replays the journal and acts on it. The decision is a
    first-class action: decision in {approve, deny, steer}."""
    append(world, "approval_decision", {"idx": idx, "decision": decision, "correction": correction})
    extra = f" (correction: ${correction['amount']:.2f})" if correction else ""
    print(f"    [human] decision = {decision}{extra}")
    run(world)


print("resume ready (records the decision, then replays + continues).")

## Step 6 - Streaming progress: a live feed is just a read over the journal

Because every step is a journal event, a progress feed needs no new machinery: `stream` walks the same log and renders each event as a human-readable line. The pause, the human decision, and the resumed steps all show up, because they are all events. This is the payoff of "state is the fold over the log": observability comes for free from the same artifact that makes the run durable.

In [ ]:
def stream(world):
    for e in world["journal"]:
        d, t = e["data"], e["type"]
        if t == "run_started":
            print("      > run started")
        elif t == "llm_decided":
            print(f"      > deciding: {d['tool']}")
        elif t == "tool_result":
            print(f"      > done: {d['result']}")
        elif t == "pending_approval":
            print(f"      > PAUSED for approval ({d['reason']}); token {d['token']}")
        elif t == "approval_decision":
            print(f"      > human decision: {d['decision']}")
        elif t == "finished":
            print(f"      > finished: {d['answer']}")


print("stream ready (a live feed is just a read over the journal).")

## generate(): the real LLM path (reference shape only)

Same device as Parts 1-9. Offline, the deterministic plan is the source of truth for everything in the output; the demo never calls `generate()`. On the real path the controller would decide each step, and the gate/pause/resume machinery would wrap it unchanged. SDK names and model IDs move fast, so only `generate()` would need edits to go live.

In [ ]:
def generate(prompt):
    """REAL path: ask a hosted LLM for the next step. Unused offline."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",                        # a small chat model; check names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

# Anthropic / Claude alternative. Swap in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return resp.content[0].text


if os.environ.get("OPENAI_API_KEY"):
    print("[controller] OPENAI_API_KEY set; the real LLM would drive the loop via generate(). "
          "Falling through to the deterministic plan for reproducibility.")
else:
    print("[controller] no OPENAI_API_KEY; using the deterministic plan (offline default)")

## Step 7 - A top-level helper that catches the interrupt

`_pause` is the top level that runs the agent and CATCHES `PendingApproval`. This is the spot Step 2 insisted on: the exception is handled, the world (with its just-journaled `pending_approval`) is persisted, and the token is returned to the caller. The kernel never dies. If the run somehow completed without gating, the helper says so and returns `None`.

In [ ]:
def _pause(world, label):
    try:
        run(world)
        print(f"    [{label}] (completed without pausing)")
        return None
    except PendingApproval as p:                       # caught at the top level, NOT sys.exit
        print(f"    PAUSED: {p.reason}")
        print(f"    returned token {p.token!r}; the run is persisted, not dead. Ledger so far: "
              f"{world['ledger'] or '(empty, no money moved yet)'}")
        return p.token


print("_pause ready (catches PendingApproval; the kernel stays alive).")

## Demo - run to the gate and pause, then resolve it three ways

Everything below runs fully offline and mirrors a real deployment's two processes.

**Phase 1 - run until the gate, then PAUSE.** The first invocation runs step 0 (search), reaches the gated $180 refund, journals `pending_approval`, and raises `PendingApproval`. The top-level `_pause` catches it (NOT `sys.exit`, so the kernel survives) and returns token `appr-1`. The ledger is empty: unlike Part 9's crash, the pause happens BEFORE the side effect, so no money has moved. We then stream the journal to show the pause as a feed.

**Phase 2 - three ways a human resolves the pause**, each from a DEEP COPY of the same paused world so they stay independent:
- **APPROVE** -> replay memoizes step 0, the gate sees the approval, the refund executes (ledger `$180.00`), and the run finishes. The stable key keeps a retried resume effectively-once.
- **DENY** -> the gate sees the denial, journals a refusal and `finished`, and no money moves (ledger empty).
- **STEER** -> the approver lowers the refund to `$90.00`, now under the threshold, so it proceeds (ledger `$90.00`). The steer branch streams its full timeline: pause, decision, and resume, all from the one journal.

In [ ]:
bar = "=" * 72
print(bar)
print("PAUSE, APPROVE, RESUME, STEER  -  human-in-the-loop on a durable agent")
print(bar)
if os.environ.get("OPENAI_API_KEY"):
    print("[controller] OPENAI_API_KEY set; the real LLM would drive the loop via generate(). "
          "Falling through to the deterministic plan for reproducibility.")
else:
    print("[controller] no OPENAI_API_KEY; using the deterministic plan (offline default)")
print(f"\nApproval threshold: ${THRESHOLD:.0f}. The task is a ${STEPS[1][1]['amount']:.0f} refund "
      "for ORD-3300, which is over the threshold.")

# --- Phase 1: run until the gate, then pause. ---
print("\n" + "-" * 72)
print("RUN until the approval gate, then PAUSE (interrupt -> PendingApproval).")
print("-" * 72)
paused = new_world()
token = _pause(paused, "run")
print("\n    streaming progress from the journal:")
stream(paused)

In [ ]:
print("\n" + bar)
print("RESUME A) APPROVE: the gated refund executes (effectively-once via its key).")
print(bar)
wa = copy.deepcopy(paused)
resume(wa, "approve")
print(f"    ledger: {wa['ledger']}")

In [ ]:
print("\n" + bar)
print("RESUME B) DENY: no money moves; the run finishes with a refusal.")
print(bar)
wb = copy.deepcopy(paused)
resume(wb, "deny")
print(f"    ledger: {wb['ledger'] or '(empty)'}")

In [ ]:
print("\n" + bar)
print("RESUME C) STEER: the approver lowers the refund to $90, under the threshold.")
print(bar)
wc = copy.deepcopy(paused)
resume(wc, "steer", correction={"amount": 90.0})
print(f"    ledger: {wc['ledger']}")
print("\n    streaming the STEER timeline (pause + decision + resume, all from the journal):")
stream(wc)

print("\n" + bar)
print("Done. A durable agent can now hand control to a human and come back:")
print("  - interrupt() raises a serializable PendingApproval (a token), NOT sys.exit")
print("  - resume() replays the journal and acts on the decision (effectively-once)")
print("  - the decision is a first-class action: APPROVE, DENY, or STEER a correction")
print("  - progress streams from the same journal events that make it durable")
print(bar)

## Wrap-up

Part 9 made the agent durable: an append-only journal, deterministic replay with memoization, and idempotency keys for effectively-once side effects. But a durable run still went start to finish on its own, with no way to stop for a human or be corrected mid-flight.

Part 10 added a human-in-the-loop checkpoint on top of that same journal, with four moves:

- **interrupt()** journals a `pending_approval` event with a token and raises a serializable `PendingApproval` that is caught at the TOP LEVEL, never `sys.exit`. The host process stays alive and the pause is serializable, so it can cross a process boundary on disk.
- **resume()** records the human's decision as a first-class journal event, then replays the journal and continues. The inherited idempotency key keeps an approved resume effectively-once even under retry.
- **STEER** makes the decision a first-class action beyond approve/deny: the human can inject a correction (here, lower the refund under the threshold), which is the same mechanism that answers a clarifying question back to the user.
- **Streaming** falls out for free: because every step is a journal event, a live progress feed, including the pause and the decision, is just a read over the log.

Remember the scope, as in Part 9: "cross-process resume" means re-entering `resume()` on the same surviving journal, the artifact that would live on disk; the deterministic plan is what makes the run byte-reproducible offline.

**Part 11 - the observability capstone** is next. The same journal that made the agent durable (Part 9) and pausable (Part 10) becomes the backbone for spans, traces, and metrics: once every step is already an event, turning the log into a full observability view is the natural last step.